In [23]:
#%pip install geopandas pyogrio

In [24]:
from pathlib import Path

import pandas as pd
import geopandas as gpd

In [25]:
PROJECT_DIR = Path("/Users/enrue/Developer/jupyter-setup/cip_projekt")
input_file = (PROJECT_DIR / "data_processed" / "flatfox_listings_merged.csv")
output_file = (PROJECT_DIR / "data_processed" / "flatfox_rentals_zentralschweiz.csv")

# Beide bisher genannten Dateinamen berücksichtigen.
gpkg_names = [
    "swissBOUNDARIES3D_1_5_LV95_LN02.gpkg"
]

gpkg_path = next(
    (
        PROJECT_DIR / "geodata" / name
        for name in gpkg_names
        if (PROJECT_DIR / "geodata" / name).is_file()
    ),
    None,
)

if gpkg_path is None:
    available = list((PROJECT_DIR / "geodata").glob("*.gpkg"))
    raise FileNotFoundError(
        f"Keine der erwarteten GeoPackage-Dateien gefunden. "
        f"Vorhandene Dateien: {available}"
    )

print(f"Verwendete Kantonsgrenzen: {gpkg_path.name}")

df = pd.read_csv(
    input_file,
    encoding="utf-8-sig",
    dtype="string",
    keep_default_na=False,
    na_values=[""],
)

# Filter der Rohdaten
mask = (df["offer_type"].eq("RENT") & df["object_category"].isin(["APARTMENT", "HOUSE"])).fillna(False)

df_rentals_geo = df.loc[mask].copy().reset_index(drop=True)


# Kantonsflächen laden
layers = gpd.list_layers(gpkg_path)

canton_layers = layers.loc[
    layers["name"].str.contains(
        "kantonsgebiet", case=False, na=False
    ),
    "name",
].tolist()

if len(canton_layers) != 1:
    raise ValueError(
        f"Kantons-Layer nicht eindeutig gefunden: "
        f"{layers['name'].tolist()}"
    )

cantons = gpd.read_file(gpkg_path, layer=canton_layers[0])

cantons = cantons.rename(columns={
    column: column.upper()
    for column in cantons.columns
    if column != cantons.geometry.name
})

if cantons.geometry.name != "geometry":
    cantons = cantons.rename_geometry("geometry")

cantons = cantons.loc[cantons["ICC"].eq("CH")].copy()

canton_codes = dict(enumerate(
    [
        "ZH", "BE", "LU", "UR", "SZ", "OW", "NW", "GL", "ZG",
        "FR", "SO", "BS", "BL", "SH", "AR", "AI", "SG", "GR",
        "AG", "TG", "TI", "VD", "VS", "NE", "GE", "JU",
    ],
    start=1,
))

cantons["kanton_code"] = pd.to_numeric(cantons["KANTONSNUMMER"], errors="coerce").map(canton_codes)

if cantons["kanton_code"].isna().any():
    raise ValueError("Unbekannte Kantonsnummer in den Geodaten.")


# Inserat-Koordinaten vorbereiten
longitude = pd.to_numeric(df_rentals_geo["longitude"], errors="coerce")
latitude = pd.to_numeric(df_rentals_geo["latitude"], errors="coerce")

valid_coordinates = (longitude.between(-180, 180) & latitude.between(-90, 90)).fillna(False)

df_rentals_geo["kanton_geo"] = pd.Series(pd.NA,index=df_rentals_geo.index,dtype="string",)
df_rentals_geo["kanton_geo_status"] = "koordinaten_ungueltig"


# Kantone anhand der Koordinaten bestimmen
if valid_coordinates.any():
    points = gpd.GeoDataFrame(
        index=df_rentals_geo.index[valid_coordinates],
        geometry=gpd.points_from_xy(
            longitude.loc[valid_coordinates],
            latitude.loc[valid_coordinates],
        ),
        crs="EPSG:4326",
    ).to_crs(cantons.crs)

    matches = gpd.sjoin(
        points,
        cantons[["kanton_code", "geometry"]],
        how="left",
        predicate="intersects",
    )

    candidates = matches.groupby(level=0)["kanton_code"].agg(
        lambda values: tuple(sorted(set(values.dropna())))
    )

    assigned = candidates.map(
        lambda values: values[0] if len(values) == 1 else pd.NA
    )

    df_rentals_geo["kanton_geo"] = (
        assigned.reindex(df_rentals_geo.index).astype("string")
    )

    df_rentals_geo.loc[
        valid_coordinates, "kanton_geo_status"
    ] = "kein_treffer"

    ambiguous_indices = candidates[
        candidates.map(len).gt(1)
    ].index

    df_rentals_geo.loc[
        ambiguous_indices, "kanton_geo_status"
    ] = "mehrdeutiger_treffer"

    df_rentals_geo.loc[
        df_rentals_geo["kanton_geo"].notna(),
        "kanton_geo_status",
    ] = "zugeordnet"


# Auf die Zentralschweiz eingrenzen
central_cantons = ["LU", "UR", "SZ", "OW", "NW", "ZG"]

df_central = df_rentals_geo.loc[df_rentals_geo["kanton_geo"].isin(central_cantons)].copy()

# Ergebnis als CSV speichern
output_file.parent.mkdir(parents=True, exist_ok=True)

df_central.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig",
)

print(f"\nZusammengeführte Inserate: {len(df)}")
print(f"Mietwohnungen und Häuser: {len(df_rentals_geo)}")
print(f"Davon Zentralschweiz: {len(df_central)}")
print(f"CSV gespeichert: {output_file}")

print("\nStatus der Kantonszuordnung:")
display(df_rentals_geo["kanton_geo_status"].value_counts())

print("\nInserate pro Zentralschweizer Kanton:")
display(
    df_central["kanton_geo"]
    .value_counts()
    .reindex(central_cantons, fill_value=0)
)

Verwendete Kantonsgrenzen: swissBOUNDARIES3D_1_5_LV95_LN02.gpkg

Zusammengeführte Inserate: 3351
Mietwohnungen und Häuser: 1554
Davon Zentralschweiz: 868
CSV gespeichert: /Users/enrue/Developer/jupyter-setup/cip_projekt/data_processed/flatfox_rentals_zentralschweiz.csv

Status der Kantonszuordnung:


kanton_geo_status
zugeordnet    1554
Name: count, dtype: int64


Inserate pro Zentralschweizer Kanton:


kanton_geo
LU    587
UR     19
SZ    110
OW     22
NW     12
ZG    118
Name: count, dtype: Int64